# Lektion 18: Säkerställa AI-agenter med kryptografiska kvitton

## Praktisk anteckningsbok

Den här anteckningsboken går igenom fyra övningar:

1. **Signera ditt första kvitto** för ett agentverktygsanrop och verifiera det.
2. **Manipulera kvittot** och se verifieringen misslyckas.
3. **Bygg en kedja med tre kvitton** och bekräfta kedjans integritet.
4. **Inslut ett Microsoft Agent Framework-verktygsanrop** så att varje åtgärd avger ett kvitto.

Alla kryptografiska primitiva importeras från väl underhållna bibliotek (`pynacl` för Ed25519, `jcs` för RFC 8785 kanonisk JSON, `hashlib` från Python standardbibliotek för SHA-256). Själva kvittologistiken är vanlig Python som du kan läsa och modifiera.

Kör cellerna i ordning. Varje avsnitt är kort och självständigt.


## Installation

Installera de två beroenden. Båda har tillåtande licenser (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Hjälpverktyg

Dessa två hjälpverktyg hanterar base64url-kodning (utan padding) och SHA-256-hashning av godtyckliga objekt. De håller resten av anteckningsboken fokuserad på själva kvittologiken.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Avsnitt 1: Signera ditt första kvitto

Föreställ dig att vår agent för **Contoso Travel** precis har sökt flyg från Sydney till Los Angeles för en kund. Vi vill spela in gatewayens uttalande om detta verktygsanrop så att en framtida revisor kan verifiera det under en separat fastställd utgivarnyckel.

### Steg 1.1: Generera en signeringsnyckel

I produktion skulle agentens signeringsnyckel finnas i en hårdvarusäkerhetsmodul (HSM), Azure Key Vault eller en liknande skyddad lagring. För denna lektion genererar vi en ny nyckel i minnet.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Steg 1.2: Skapa kvittots nyttolast

Nyttolasten innehåller allt vi vill att kvittot ska intyga: vem som agerade, vilket verktyg, med vilka argument, vad som kom tillbaka, under vilken policy, och när. Vi hashar argumenten och resultatet istället för att inkludera dem direkt för att kvittot inte ska läcka känsligt innehåll.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Steg 1.3: Signera och montera kvittot

Tre steg:

1. Kanonisera nyttolasten med JCS så att två implementationer som producerar samma logiska kvitto ger byte-identiska bytes.
2. Signera de kanoniska bytesen direkt med Ed25519:s privata nyckel. PureEdDSA hashar meddelandet internt, så en extra för-hash skulle ändra protokollet.

Signaturen bifogas sedan den ursprungliga nyttolasten för att producera det slutgiltiga kvittot.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Steg 1.4: Verifiera kvittot

Verifiering vänder processen. Vi tar bort signaturen, beräknar om de kanoniska byten, och kontrollerar signaturen mot utfärdarens offentliga nyckel.

Verifiering sker offline: ingen tjänst att anropa och ingen nätverksåtkomst. Det krävs ändå en sak utöver kvittot: utfärdarens offentliga nyckel, erhållen genom en kanal som revisor redan litar på, såsom en JWK-set som utfärdaren publicerar eller ett värde som har fastställts i förväg. Fältet `public_key` inuti kvittot kan inte fylla den rollen eftersom den som skriver kvittot också skriver det fältet. Avsnitt 2 visar förfalskningen detta förhindrar.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Du bör se `Receipt is valid: True`, `Pre-hashed receipt valid: False` och `Malformed receipts rejected: True`. Det positiva fallet bevisar att signeringsvägen med direkt-JCS fungerar; den negativa kontrollen gör signaturscope-regeln exekverbar istället för att bara vara prosa; sista raden visar att felaktig inmatning returnerar `False` istället för att kasta ett undantag.


## Avsnitt 2: Manipulera kvittot

Hela poängen med kvitton är att de ska vara manipuleringssäkra. Låt oss bevisa det.

Vi kommer att ändra exakt ett tecken i kvittot och se att verifieringen misslyckas.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Vad hände precis?

När vi ändrade `policy_id` ändrades de kanoniska bytena. Signaturen (som var över de ursprungliga kanoniska bytena) matchar inte längre. Verifieringen returnerar korrekt `False`.

Det finns inget sätt att ändra ett signerat fält, alltså något utanför `signature`-objektet, och ändå få kvittot att verifieras mot utfärdarens offentliga nyckel, om inte angriparen har utfärdarens privata nyckel. Själva `signature`-objektet är inte signerat: `sig` är signaturen, och `verify_receipt` ignorerar `public_key`, så att redigera det fältet enbart ändrar ingenting. Så länge den privata nyckeln finns i ett nyckelförråd och verifierare använder utfärdarens oberoende publicerade nyckel är en ändring i ett signerat fält omöjlig att dölja. Den förutsättningen är viktig: nästa cell visar ett modifierat kvitto som verifieras perfekt mot en annan nyckel.

Prova själv: ändra `tool_name` eller `agent_id` eller `timestamp` i cellen ovan och kör igen. Varje ändring i ett signerat fält ger ett ogiltigt kvitto.


### Manipulationen som signaturen inte upptäcker

Angriparen ovan signerade inte om. En mer försiktig angripare skulle: ändra kvittot,
signera det med en nyckel de själva genererat, och lägga den nyckeln i
`public_key` fältet. Resultatet är ett kvitto vars signatur är helt äkta
för dess innehåll.

Kör detta och se vilken av de två kontrollerna som fångar det.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Du bör se `True` sedan `False`. De tre `assert` raderna gör detta till en
kontroll: cellen ger ett fel om förfalskningen någonsin verifieras mot
utgivarens nyckel.

Förfalskningen är internt konsekvent. Dess signatur överensstämmer verkligen med dess
innehåll och den nyckel den bär, så en verifierare som läser nyckeln från
kvittot rapporterar den som giltig. Ingenting i kvittot ser fel ut, eftersom
ingenting är inkonsekvent.

Det enda som avslöjar det är att kontrollera mot en nyckel som angriparen inte hade någon
möjlighet att påverka. Detta är varför `verify_receipt` tar nyckeln som ett argument: en
funktion med signaturen `verify_receipt(receipt)` kan inte skrivas säker,
eftersom den enda nyckeln den kan nå är den som en angripare tillhandahållit.

Samma kvitto sparas som `sample_receipts/04_resigned_receipt.json` om du
vill inspektera det utanför anteckningsboken.


## Avsnitt 3: Knyt kvitton i kedja

Ett enda kvitto skyddar en handling. De flesta agenter utför många handlingar. För att göra hela sekvensen manipulationssäker länkar vi varje kvitto till det föregående genom att inkludera det föregående kvittots hash i det nya kvittots payload.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Om någon tar bort eller ändrar ordningen på ett kvitto i den presenterade sekvensen bryts nästa länk. Att upptäcka trunkering eller signaturkontrollerad omskrivning av suffix kräver också ett fastställt aktuellt huvud, extern kontrollpunkt eller autentiserat befolkningsåtagande.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Bryt nu kedjan genom att manipulera det mittersta kvittot och verifiera igen. Det manipulerade kvittot misslyckas med sin signaturkontroll, OCH nästa kvitto misslyckas med sin kedjelänkskontroll (eftersom dess `previous_receipt_hash` inte längre matchar det modifierade mittersta kvittots hash).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kvitto 0 verifieras fortfarande (det ändrades inte och har ingen föregångare att förlita sig på). Kvitto 1 misslyckas med sin signaturkontroll eftersom vi ändrade `tool_args_hash`. Kvitto 2 misslyckas med sin kedjelänkcheck eftersom dess `previous_receipt_hash` beräknades mot det ursprungliga (nu ändrade) kvitto 1.

Även om en angripare signerar om det ändrade kvitto 1, misslyckas den nya signaturen mot utfärdarens nyckel såvida de inte innehar utfärdarens privata nyckel (en nysignering med deras egen nyckel misslyckas, som Sektion 2 visade), och kedjelänksavvikelsen i kvitto 2 skulle fortfarande avslöja manipuleringen. För att dölja ändringen skulle angriparen behöva signera om varje kvitto från ändringspunkten och framåt, vilket kräver innehav av utfärdarens privata nyckel.


## Avsnitt 4: Wrappa ett agentverktygsanrop med kvittosignering

Vid en verklig implementering vill du inte att varje agentförfattare ska komma ihåg att anropa `make_receipt`. Du vill att kvittosignering ska ske automatiskt för varje verktygsanrop.

Här är det enklaste mönstret: en wrapper-klass som tar vilken som helst anropbar verktygsfunktion och returnerar en kvittoutgivande version av den. Detta anpassar sig till vilket agentramverk som helst, inklusive Microsoft Agent Framework (`agent_framework.foundry`).

Om du inte har ett Microsoft Foundry-projekt uppsatt demonstrerar den lokala mocken nedan ändå mönstret.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrera med Microsoft Agent Framework

Wrappen `ReceiptedTool` ovan är ramverksoberoende. För att använda den i en agent byggd med Microsoft Agent Framework, registrera den wrappade funktionen som ett verktyg. En skiss (du skulle ersätta mocket med en riktig Microsoft Foundry-verktygsregistrering):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agentramverket behöver inte veta något om kvitton. Kvitto-signering är inpackad runt verktyget, inte inbyggd i ramverket. Så här lägger du till ursprung i befintlig agentkod utan att skriva om agenten.


## Sammanfattning och stretch-utmaning

Du har:

- Genererat ett Ed25519-nyckelpar.
- Skapat och signerat ett kvitto för ett agentverktygsanrop.
- Verifierat kvittot offline med endast utgivarens offentliga nyckel, erhållen oberoende av kvittot.
- Manipulerat ett kvitto och observerat verifieringsfel.
- Skapat en hash-kedjad sekvens av tre kvitton.
- Manipulerat mitten av kedjan och observerat både signaturfel och kedjelänksfel.
- Omslutit en verktygsfunktion med automatisk kvittosignering.

**Stretch-utmaning.** Utöka kvittoschemat med ett `request_id`-fält (en UUID för distribuerad spårning). Uppdatera `make_receipt` för att inkludera detta, och bekräfta att kvitton fortfarande verifieras från början till slut. Modifiera sedan fältet efter signering och bekräfta att verifiering misslyckas. Detta tvingar dig att internalisera hur varje byte av den kanoniska kodningen bidrar till signaturen.

**Viktig gräns.** Under en fast nyckel och kedjekontext bevisar kvitton signaturtillhörighet, innehållsintegritet och en länk till den uppvisade föregångaren. De bevisar INTE körning, källa, betrodd tid, fullständighet, policyutvärdering eller korrekthet. Kvitton är en grund. Styrning är systemet du bygger ovanpå.

Läs lektionens README igen med den gränsen i åtanke. Det vanligaste misstaget team gör med kvitton är att anta att "vi har kvitton" betyder "vi är styrda." Det gör det inte. Kvitton gör agentbeteendet granskningsbart. De gör det inte korrekt.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
